# Notebook 3: Training Job (HPO) + Model Registry

This notebook submits a remote training job to Snowflake compute using the `@remote` decorator. The job performs hyperparameter optimization (HPO) over an Isolation Forest model for well health anomaly detection.

**Pipeline Position:** Consumes features from the Dynamic Table (Notebook 2), trains the best model via grid search, and registers it in the Model Registry for inference (Notebook 4).

**Architecture:**
- `@remote` sends the training function to `COCO_ML_COMPUTE_POOL` (Snowpark Container Services)
- HPO: Grid search over `contamination`, `n_estimators`, `max_features` (18 configurations)
- Best model logged to Snowflake Experiment Tracking with metrics
- Final model registered as `WELL_HEALTH_MODEL V2` in the Model Registry

## 1. Connect and Configure

Establish a Snowpark session and load configuration constants (database, schema, compute pool, stage, model name/version, and experiment name) from shared utilities.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DATABASE = "ENERGY_DEMO"
SCHEMA = "WELLS"
COMPUTE_POOL = "COCO_ML_COMPUTE_POOL"
STAGE = "MODELS_STAGE"
MODEL_NAME = "WELL_HEALTH_MODEL"
MODEL_VERSION = "V2"
EXPERIMENT_NAME = "WELL_HEALTH_ANOMALY_DETECTION"

session.use_database(DATABASE)
session.use_schema(SCHEMA)
print(f"Connected to {DATABASE}.{SCHEMA}")
print(f"Compute Pool: {COMPUTE_POOL}")

## 2. Define the Remote Training Function

The entire function below executes on Snowflake SPCS compute via the `@remote` decorator.

**What it does:** Loads data → HPO grid search → trains best IsolationForest → wraps in CustomModel → registers in Model Registry.

---

### What is a CustomModel?

A `CustomModel` is Snowflake ML's way to register **any arbitrary Python inference logic** in the Model Registry — not just raw sklearn/xgboost models. You subclass `snowflake.ml.model.custom_model.CustomModel`, define a `predict` method decorated with `@inference_api`, and register it with `registry.log_model()`. At inference time (via `mv.run()` or an SPCS service), Snowflake calls your `predict` method directly.

### Why we need it here

Sklearn's `IsolationForest` only gives us:
- `predict()` → binary 1 (normal) or -1 (anomaly)
- `decision_function()` → raw anomaly score (unbounded, not interpretable)

Neither is a usable health metric. We want a **0–1 score** where 0 = unhealthy and 1 = healthy.

Our `WellHealthModel(CustomModel)` wraps the trained pipeline and adds a **min-max normalization** step:
1. Stores the pipeline + the min/max of `decision_function` scores from training
2. At inference: runs `decision_function` on new data, normalizes against the training range, clips to [0, 1]
3. Returns `HEALTH_SCORE` — a single float consumers can threshold (e.g., < 0.1 = alert)

### How it flows at inference time

```
Features → CustomModel.predict() → pipeline.decision_function() → min-max normalize → HEALTH_SCORE (0-1)
```

Once registered, the model is deployed as an SPCS service. Consumers call `SERVICE!PREDICT(...)` in SQL and get back `HEALTH_SCORE` directly — no post-processing needed.

In [ ]:
from snowflake.ml.jobs import remote


@remote(
    COMPUTE_POOL,
    stage_name=STAGE,
    pip_requirements=["scikit-learn", "numpy", "pandas", "snowflake-ml-python"],
    external_access_integrations=["PYPI_ACCESS_INTEGRATION"],
    session=session,
)
def train_well_health_model(
    db: str, schema: str, experiment_name: str, model_name: str, model_version: str
):
    """Remote training job: HPO over Isolation Forest, register CustomModel with health scores."""
    import numpy as np
    import pandas as pd
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.metrics import silhouette_score
    from sklearn.model_selection import ParameterGrid
    from snowflake.snowpark import Session

    session = Session.builder.getOrCreate()
    session.use_database(db)
    session.use_schema(schema)

    print("[1/6] Loading training data...")
    df_sensors = session.sql("""
        SELECT API_NO, INTAKE_PRESSURE_PSI, DISCHARGE_PRESSURE_PSI,
            DISCHARGE_PRESSURE_PSI - INTAKE_PRESSURE_PSI AS PRESSURE_DIFFERENTIAL,
            MOTOR_TEMP_F, MOTOR_AMPS, VIBRATION_IPS,
            WELLHEAD_PRESSURE_PSI, WELLHEAD_TEMP_F, FREQUENCY_HZ
        FROM WELL_SENSORS WHERE READING_TS >= '2025-06-01'
    """).to_pandas()

    df_prod = session.sql("""
        SELECT API_NO,
            AVG(OIL_BBL) AS AVG_OIL, AVG(GAS_MCF) AS AVG_GAS,
            AVG(WATER_BBL) AS AVG_WATER, AVG(RUNTIME_HOURS) AS AVG_RUNTIME,
            AVG(WATER_BBL) / NULLIF(AVG(OIL_BBL) + AVG(WATER_BBL), 0) AS AVG_WATER_CUT,
            AVG(GAS_MCF) / NULLIF(AVG(OIL_BBL), 0) AS AVG_GOR
        FROM WELL_PRODUCTION WHERE PRODUCTION_DATE >= '2025-06-01'
        GROUP BY API_NO
    """).to_pandas()

    sensor_cols = [
        "INTAKE_PRESSURE_PSI",
        "DISCHARGE_PRESSURE_PSI",
        "PRESSURE_DIFFERENTIAL",
        "MOTOR_TEMP_F",
        "MOTOR_AMPS",
        "VIBRATION_IPS",
        "WELLHEAD_PRESSURE_PSI",
        "WELLHEAD_TEMP_F",
        "FREQUENCY_HZ",
    ]
    prod_cols = [
        "AVG_OIL",
        "AVG_GAS",
        "AVG_WATER",
        "AVG_RUNTIME",
        "AVG_WATER_CUT",
        "AVG_GOR",
    ]
    feature_cols = sensor_cols + prod_cols

    df = df_sensors.merge(df_prod, on="API_NO", how="left").fillna(0)
    X = df[feature_cols].values
    print(f"  Training data: {X.shape[0]:,} x {X.shape[1]} features")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    print("[2/6] Running HPO (grid search)...")
    param_grid = {
        "n_estimators": [100, 200, 300],
        "contamination": [0.03, 0.05, 0.08],
        "max_features": [0.8, 1.0],
    }
    best_score, best_params = -1, None
    results = []

    for params in ParameterGrid(param_grid):
        model = IsolationForest(**params, random_state=42, n_jobs=-1)
        model.fit(X_scaled)
        preds = model.predict(X_scaled)
        score = silhouette_score(X_scaled, preds, sample_size=min(5000, len(preds)))
        results.append({**params, "silhouette_score": score})
        if score > best_score:
            best_score, best_params = score, params

    print(f"  Best: {best_params}, silhouette={best_score:.4f}")

    print("[3/6] Building pipeline with scaler + best model...")
    pipeline = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "isolation_forest",
                IsolationForest(**best_params, random_state=42, n_jobs=-1),
            ),
        ]
    )
    pipeline.fit(X)

    raw_scores = pipeline.decision_function(X)
    predictions = pipeline.predict(X)
    n_anomalies = (predictions == -1).sum()
    anomaly_pct = n_anomalies / len(predictions) * 100

    score_min = float(raw_scores.min())
    score_max = float(raw_scores.max())
    print(f"  Decision function range: [{score_min:.4f}, {score_max:.4f}]")

    print("[4/6] Building CustomModel with min-max health score...")
    import pickle
    import json
    import tempfile
    import os
    from snowflake.ml.model import custom_model

    tmpdir = tempfile.mkdtemp()
    pipeline_path = os.path.join(tmpdir, "pipeline.pkl")
    bounds_path = os.path.join(tmpdir, "score_bounds.json")
    with open(pipeline_path, "wb") as f:
        pickle.dump(pipeline, f)
    with open(bounds_path, "w") as f:
        json.dump({"min": score_min, "max": score_max}, f)

    model_context = custom_model.ModelContext(
        artifacts={
            "pipeline": pipeline_path,
            "score_bounds": bounds_path,
        }
    )

    class WellHealthModel(custom_model.CustomModel):
        def __init__(self, context: custom_model.ModelContext) -> None:
            super().__init__(context)
            import pickle
            import json

            with open(context.path("pipeline"), "rb") as f:
                self.pipeline = pickle.load(f)
            with open(context.path("score_bounds"), "r") as f:
                bounds = json.load(f)
            self.score_min = bounds["min"]
            self.score_max = bounds["max"]

        @custom_model.inference_api
        def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
            raw = self.pipeline.decision_function(input_df.values)
            health_scores = (raw - self.score_min) / (self.score_max - self.score_min)
            health_scores = np.clip(health_scores, 0.0, 1.0)
            return pd.DataFrame({"HEALTH_SCORE": np.round(health_scores, 4)})

    well_health_model = WellHealthModel(model_context)
    sample_input = pd.DataFrame(X[:5], columns=feature_cols)
    test_output = well_health_model.predict(sample_input)
    print(f"  Test health scores: {test_output['HEALTH_SCORE'].tolist()}")

    print("[5/6] Logging to Experiment Tracking...")
    from snowflake.ml.experiment import ExperimentTracking

    exp = ExperimentTracking(session=session)
    exp.set_experiment(experiment_name)

    import datetime

    run_name = f"hpo_run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    with exp.start_run(run_name):
        exp.log_params(
            {
                **{k: str(v) for k, v in best_params.items()},
                "n_features": str(len(feature_cols)),
                "n_samples": str(X.shape[0]),
                "model_type": "CustomModel(IsolationForest+MinMaxHealth)",
                "score_min": str(round(score_min, 4)),
                "score_max": str(round(score_max, 4)),
            }
        )
        exp.log_metrics(
            {
                "silhouette_score": round(best_score, 4),
                "anomaly_pct": round(anomaly_pct, 2),
            }
        )

    print("[6/6] Registering CustomModel in Model Registry...")
    from snowflake.ml.registry import Registry

    registry = Registry(session=session)
    version = f"{model_version}_{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"
    registry.log_model(
        well_health_model,
        model_name=model_name,
        version_name=version,
        sample_input_data=sample_input,
        pip_requirements=["scikit-learn", "numpy", "pandas"],
        comment=f"CustomModel(Scaler+IsolationForest+MinMaxHealth). HPO: {best_params}. Silhouette: {best_score:.4f}. Min-max normalized 0-1 health score.",
    )
    print(f"  Registered: {model_name} {version}")
    print("  Output: HEALTH_SCORE (0.0=most anomalous, 1.0=most healthy)")

    return {
        "status": "success",
        "best_params": best_params,
        "silhouette_score": best_score,
        "anomaly_pct": anomaly_pct,
        "configs_tested": len(results),
        "version": version,
        "output": "HEALTH_SCORE (0.0-1.0, min-max normalized)",
    }

## 3. Submit the Job

Send the training function to `COCO_ML_COMPUTE_POOL` for remote execution. The `@remote` decorator returns a job handle that we can poll for status. Typical runtime is ~2-3 minutes.

In [ ]:
print(f"Submitting to {COMPUTE_POOL}...")
job = train_well_health_model(
    DATABASE, SCHEMA, EXPERIMENT_NAME, MODEL_NAME, MODEL_VERSION
)

print("Job submitted! Waiting for completion...")
job.wait()
print(f"Status: {job.status}")

## 4. View Results

Check job status and print the returned metrics dictionary (best parameters, silhouette score, number of anomalies detected). If the job failed, print the container logs for debugging.

In [ ]:
if job.status == "DONE":
    result = job.result()
    for k, v in result.items():
        print(f"  {k}: {v}")
    print(f"\nModel: {MODEL_NAME} {MODEL_VERSION}")
    print(f"Experiment: {EXPERIMENT_NAME}")
else:
    print("Job failed:")
    print(job.get_logs())